In [ ]:
import jax
jax.config.update("jax_platform_name", "cpu")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from copy import deepcopy

from gould_2026.estimator import  ArrayWithTime, CenteringEstimator, Pipeline
from gould_2026.dimension_reduction.prosvd import proSVD
from tqdm.notebook import tqdm
from gould_2026.stim_regressor import StimRegressor, StimAutoReg
from gould_2026.datasets import Daie21Dataset
from gould_2026.plotting import Palette, paper_plot_context, LINEWIDTH, make_boxplot, make_latent_quiver, make_pred_heatmap_after_quiver
import seaborn as sns
from gould_2026.save_to_cache import save_to_cache
from scipy.stats import wilcoxon
from gould_2026.utils import angle_between, rotation_matrix, check_same
import mantel_test

rng = np.random.default_rng(0)


In [ ]:
output = '/home/jgould/Downloads/daie21_1step.svg'

In [ ]:
output = None
output2 = None
output3_response_pairwise_plot = None
output4_quiver_plot = None
output5_mantel_hist = None
output6_inferred_stim_response_map = None


In [ ]:

@save_to_cache('daie_srs','/mnt/data/gould_2026_cache/')
def f(rng):

    d = Daie21Dataset(shuffle_random_state=rng.integers(0, 2**32))
    neural_data = d.neural_data

    stim_t = []
    offsets = []
    for s in d.stimuli:
        x = neural_data.slice_by_time(slice(s.t-neural_data.dt*4, s.t+neural_data.dt*4 ))
        i = -(np.argmax(np.diff(x.mean(axis=1)))- 3)
        offsets.append(i)
    stim = ArrayWithTime(d.stimuli, d.stimuli.t - np.array(offsets) * neural_data.dt)


    p = Pipeline([
        CenteringEstimator(init_size=100, nan_when_uninitialized=True),
        pro := proSVD(k=10),
    ])

    latents = p.offline_run_on(neural_data)






    sr1 = StimRegressor(error_on_missed_stim=False, stim_delay=2*latents.dt, log_level=2, heed_stimuli=False, attempt_correction=False)

    sr2 = StimRegressor(error_on_missed_stim=False, stim_delay=2*latents.dt, log_level=2, heed_stimuli=True, attempt_correction=True)
    sr2.stim_autoreg = StimAutoReg(n_steps_to_consider=7)


    sr1.offline_run_on([(latents,'X'),(stim,'stim')])

    stim_shifted = deepcopy(stim)
    stim_shifted.t = stim_shifted.t - 2 * latents.dt
    sr2.offline_run_on([(latents,'X'), (stim_shifted, 'stim')])

    srs = {
        'blind': sr1,
        'reg': sr2,
    }
    return srs, neural_data, stim_shifted, latents, stim, d

srs, neural_data, stim_shifted, latents, stim, d = f(rng)

In [ ]:
%matplotlib inline


colors = {
    'blind': Palette.blind,
    'reg': Palette.stim_regressed,
}

new_names = {
    'blind': 'Unaware of stimulation',
    'reg': 'Correcting for stimulation',
}


fig, ax = plt.subplots(constrained_layout=True, figsize=(4,4))
for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(norm_error.t, norm_error, '-', color=colors[k], label=f'{new_names[k]}') # f'mse_whole={np.nanmean(norm_error ** 2):.2f} mse[800:]={np.nanmean(norm_error.slice_by_time(slice(800,None))** 2):.2f}'



ax.legend(frameon=False)

ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel('Prediction error', fontsize=14)

ax.set_xlim(950, 959)

for t in stim_shifted.t:
    ax.axvspan(t - 0.1, t + .1, color='r', alpha=.2)

if output is not None:
    fig.savefig(output)



In [ ]:

stim_df = pd.DataFrame()
for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    pre_df = []
    for i, s in enumerate(stim_shifted):
        dt = neural_data.dt
        e = norm_error.slice_by_time(slice(s.t + 1*dt, s.t + 11*dt))
        pre_df.append({
            'norm_error':e,
            'stim':s,
            't':s.t,
            'group': k,
            'stim_i': i
        })

    stim_df = pd.concat([stim_df, pd.DataFrame(pre_df)], ignore_index=True)

stim_df['error'] = stim_df['norm_error'].apply(lambda x: np.sqrt(np.mean(x**2)))
stim_df['target'] = stim_df['stim'].apply(np.argmax).astype('category')


In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))

preq_errors = ArrayWithTime.from_list(srs['reg'].stim_reg.log['preq_errors'], squeeze_type='to_2d')
preq_errors = np.linalg.norm(preq_errors,axis=1)
n_observed = srs['reg'].stim_reg.n_observed

sreg = srs['reg'].stim_reg
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
error_scatter = ax.scatter(sreg.input_histories[0][:n_observed,0], sreg.input_histories[0][:n_observed,1], c=preq_errors, cmap='plasma')
fig.colorbar(error_scatter)


In [ ]:
from scipy.spatial.distance import pdist, squareform

m = squareform(pdist(sreg.output_history[:sreg.n_observed], metric=lambda x,y: angle_between(x,y)))
plt.matshow(m, cmap='plasma', vmin=0, vmax=150)

if output3_response_pairwise_plot is not None:
    plt.savefig(output3_response_pairwise_plot)

In [ ]:
idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
# assert np.linalg.norm(latents.slice(idx) - srs['reg'].stim_reg.input_histories[0][0]) < 1e-10
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1



latent_deltas = (latents).slice(idx + 6) - (latents).slice(idx)
mean_delta = latent_deltas.mean(axis=0)
mean_angle = np.atan2(mean_delta[1], mean_delta[0])
s = np.eye(mean_delta.size)
s[:2,:2] = rotation_matrix(mean_angle)


_, _, cmap_f = Palette.draelos_25_cmap()


with paper_plot_context():
    latent_0 = (latents@s)[:,0]
    latent_1 = (latents@s)[:,1]
    pre_points = (latents@s).slice(idx)
    latent_deltas = (latents@s).slice(idx + 6) - (latents@s).slice(idx)

    fig, ax = make_latent_quiver(latent_0, latent_1, pre_points, latent_deltas, cmap_f)

    quiver_axis = ax.axis()

    if output4_quiver_plot is not None:
        fig.savefig(output4_quiver_plot)


In [ ]:
quiver_axis

In [ ]:
sr = srs['reg']

def predict_from_learned_reg(to_predict, stim_reg=sr.stim_reg, stims_nan=False, stim=stim):
    predictions = []

    for row in tqdm(to_predict):
        inner_predictions = []
        for s in stim:
            pred = stim_reg.predict([row, s, row.t])
            inner_predictions.append(pred)

        inner_predictions = np.array(inner_predictions)

        if stims_nan:
            if row.t in d.stimuli:
                inner_predictions = inner_predictions * np.nan

        predictions.append(inner_predictions)
    return np.array(predictions)


In [ ]:

center = latents.slice(~np.isnan(latents).any(axis=1)).mean(axis=0)
f = lambda x: predict_from_learned_reg(x, stim=stim.slice(slice(0, 10)))

e1 = s[:, 0]
e2 = s[:, 1]
with paper_plot_context():
    fig, ax = make_pred_heatmap_after_quiver(latent_0, latent_1, quiver_axis, f, cmap_f, center, e1, e2)
    if output6_inferred_stim_response_map is not None:
        fig.savefig(output6_inferred_stim_response_map)


In [ ]:




fig, ax = plt.subplots(figsize=np.array([1, 1])*4, constrained_layout=True)

idx = np.linalg.norm(latent_deltas, axis=1) > 10

axis_slice = slice(None)

angles = pdist(latent_deltas[idx,:][:,axis_slice], metric=angle_between)
distances = pdist(pre_points[idx,:][:,axis_slice], metric='mahalanobis')
stim_distances = pdist(stim, metric='euclidean')

x, y, slope, intercept, r = mantel_test.single_fit(
    x=distances,
    y=angles,
    confound_x=stim_distances,
    rng=rng,
    shuffle=False,
)

ax.scatter(x, y, s=1)

x = np.array(ax.get_xlim())
ax.plot(x, x*slope + intercept)


In [ ]:
# partial mantel test

test_statistic, null_samples, p = mantel_test.mantel_test(
    x=distances,
    y=angles,
    confound_x=stim_distances,
    rng=rng,
    n=100_000,
)

fig, ax = plt.subplots(figsize=np.array([4, 3]), constrained_layout=True)
ax.hist(null_samples, label='null', bins=100, density=True)
ax.axvline(test_statistic, color='r')

ax.set_title(f'Mouse partial mantel test (p$\\approx${p:.4f})')
ax.set_xlabel('partial regression r')
ax.set_ylabel('density')

print(p)

if output5_mantel_hist is not None:
    fig.savefig(output5_mantel_hist)


In [ ]:

%matplotlib inline
fig, axs = plt.subplots(constrained_layout=True, figsize=(10,5), ncols=2)

n = 9
l = .1
r = 3
time_slice = slice(stim_shifted.t[n] -l, stim_shifted.t[n] + r)

ax = axs[1]
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
# ax.plot(latents[:,0], latents[:,1], '-')
idx = np.argmin(np.linalg.norm(latents - sreg.input_histories[0][0],axis=1))
idx = latents.time_to_sample(sreg.input_histories[2][:n_observed, 0]) -1
latents_at_stims = latents.slice(idx).slice_by_time(time_slice)
ax.scatter(latents_at_stims[:,0], latents_at_stims[:,1], c='r')
latent_slice = latents.slice_by_time(time_slice)
ax.plot(latent_slice[:,0], latent_slice[:,1])

ax = axs[0]
ax.plot(latent_slice.t, latent_slice, 'k');
for t in latents_at_stims.t:
    ax.axvline(t, color='r', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('latents')


In [ ]:
%matplotlib inline


fig, ax = plt.subplots(constrained_layout=True, figsize=(10,5))
for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)

    # ax.plot(norm_error.t, norm_error, '-', color=colors[k], label=f'{new_names[k]}') # f'mse_whole={np.nanmean(norm_error ** 2):.2f} mse[800:]={np.nanmean(norm_error.slice_by_time(slice(800,None))** 2):.2f}'

for s in stim_shifted:
    e = norm_error.slice_by_time(slice(s.t, s.t + .45 ))
    c = f"C{np.argmax(s)}"
    c = f"C{int(e[0] > 5)}"
    ax.plot(e.t - s.t, e, f'{c}.-')

In [ ]:
stim_df

In [ ]:
pivot = stim_df.pivot(index=['stim_i'], columns=['group'], values='error').dropna()
pivot

In [ ]:
palette = {'reg': Palette.stim_regressed, 'blind': Palette.blind,}
with paper_plot_context():
    data = {k:stim_df.loc[stim_df['group'] == k, 'error'] for k in palette}
    fig = make_boxplot(data, palette=palette, yticks=[0, 5, 10])
    if output2 is not None:
        fig.savefig(output2)


In [ ]:
sns.scatterplot(stim_df, x='t', y='error', hue='group')